In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from collections import defaultdict
import os
import random

In [12]:
DATA_DIR = './dataset-resized/dataset-resized'
BATCH_SIZE = 32
EPOCHS = 10
TRAIN_SPLIT = 0.8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_SAVE_PATH = 'garbage_model.pth'

In [13]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),                          # losowe przycinanie zamiast centrum
    transforms.RandomHorizontalFlip(),                   # losowe odbicie poziome
    transforms.RandomVerticalFlip(),                     # losowe odbicie pionowe
    transforms.RandomRotation(15),                       # losowy obrót ±15°
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),  # zmiany kolorów
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [14]:
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [15]:
def stratified_split(dataset, train_ratio):
    """Dzieli dataset zachowując proporcje klas w obu zbiorach."""
    class_indices = defaultdict(list)
    for idx, (_, label) in enumerate(dataset.samples):
        class_indices[label].append(idx)
 
    train_indices, val_indices = [], []
    for label, indices in class_indices.items():
        random.shuffle(indices)
        split = int(len(indices) * train_ratio)
        train_indices.extend(indices[:split])
        val_indices.extend(indices[split:])
 
    return train_indices, val_indices

In [16]:
full_dataset_train = datasets.ImageFolder(DATA_DIR, transform=train_transform)
full_dataset_val = datasets.ImageFolder(DATA_DIR, transform=val_transform)
 
train_indices, val_indices = stratified_split(full_dataset_train, TRAIN_SPLIT)
 
train_dataset = torch.utils.data.Subset(full_dataset_train, train_indices)
val_dataset = torch.utils.data.Subset(full_dataset_val, val_indices)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
 
class_names = full_dataset_train.classes
print(f"Znaleziono klasy: {class_names}")
print(f"Trening: {len(train_indices)} zdjęć, Walidacja: {len(val_indices)} zdjęć")
print(f"Urządzenie: {DEVICE}\n")

Znaleziono klasy: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Trening: 7939 zdjęć, Walidacja: 1986 zdjęć
Urządzenie: cuda



In [17]:
model = models.resnet50(weights='IMAGENET1K_V1')
for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
model = model.to(DEVICE)
 
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 0.0001},  # mały lr – już wytrenowane
    {'params': model.fc.parameters(), 'lr': 0.001}         # większy lr – uczymy od zera
])
 
# Scheduler zmniejsza lr gdy accuracy przestaje rosnąć
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)

In [18]:
def train():
    best_accuracy = 0.0
 
    for epoch in range(EPOCHS):
        # --- FAZA TRENINGOWA ---
        model.train()
        running_loss = 0.0
 
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
 
        # --- FAZA WALIDACYJNA ---
        model.eval()
        correct = 0
        total = 0
 
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
 
        accuracy = 100 * correct / total
        avg_loss = running_loss / len(train_loader)
        print(f'Epoka {epoch+1}/{EPOCHS} | Strata: {avg_loss:.4f} | Accuracy: {accuracy:.2f}%', end='')
 
        # Zapisujemy tylko jeśli model jest lepszy niż poprzedni
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            print(f'  ✓ Zapisano nowy najlepszy model!')
        else:
            print()
 
        # Aktualizujemy learning rate
        scheduler.step(accuracy)
 
    print(f"\nTrening zakończony! Najlepsza accuracy: {best_accuracy:.2f}%")
    print(f"Model zapisany w: {MODEL_SAVE_PATH}")
 

In [19]:
if __name__ == '__main__':
    train()

Epoka 1/10 | Strata: 0.5923 | Accuracy: 87.97%  ✓ Zapisano nowy najlepszy model!
Epoka 2/10 | Strata: 0.3731 | Accuracy: 89.33%  ✓ Zapisano nowy najlepszy model!
Epoka 3/10 | Strata: 0.2685 | Accuracy: 90.63%  ✓ Zapisano nowy najlepszy model!
Epoka 4/10 | Strata: 0.2697 | Accuracy: 92.30%  ✓ Zapisano nowy najlepszy model!
Epoka 5/10 | Strata: 0.1946 | Accuracy: 92.55%  ✓ Zapisano nowy najlepszy model!
Epoka 6/10 | Strata: 0.1591 | Accuracy: 92.35%
Epoka 7/10 | Strata: 0.1471 | Accuracy: 92.30%
Epoka 8/10 | Strata: 0.1407 | Accuracy: 90.94%
Epoka 9/10 | Strata: 0.0922 | Accuracy: 94.51%  ✓ Zapisano nowy najlepszy model!
Epoka 10/10 | Strata: 0.0720 | Accuracy: 95.02%  ✓ Zapisano nowy najlepszy model!

Trening zakończony! Najlepsza accuracy: 95.02%
Model zapisany w: garbage_model.pth
